In [ ]:
# @title **00. Install Libraries**
%pip install geopandas rasterio scipy scikit-learn tqdm shapely osmnx networkx -q

In [ ]:
# @title **Step 01. Download Spatial Data from GitHub**

import os
import shutil
import urllib.request
import zipfile
from tqdm import tqdm

# Configuration Section
GITHUB_ZIP_URL = "https://github.com/ssakthi888/P3_GT_CHECKER/archive/refs/heads/main.zip"
TEMP_ZIP_PATH = "repo.zip"
EXTRACT_DIR = "repo_temp"
REPO_FOLDER_NAME = "P3_GT_CHECKER-main"
TARGETS_TO_MOVE = ["District","GT_Points"]

class DownloadProgressBar(tqdm):
    # Hook to update tqdm progress bar during URL retrieval
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

def download_repository():
    # Download the entire GitHub repository as a ZIP file
    print("Downloading repository data...")
    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc="Downloading") as t:
        urllib.request.urlretrieve(GITHUB_ZIP_URL, filename=TEMP_ZIP_PATH, reporthook=t.update_to)

def extract_and_organize_data():
    # Extract ZIP and move required files to the main Colab directory
    print("Extracting files...")
    with zipfile.ZipFile(TEMP_ZIP_PATH, 'r') as zip_ref:
        file_list = zip_ref.namelist()
        # Loop through files with tqdm for extraction progress tracking
        for file in tqdm(file_list, desc="Extracting"):
            zip_ref.extract(file, EXTRACT_DIR)
            
    source_dir = os.path.join(EXTRACT_DIR, REPO_FOLDER_NAME)
    
    # Move specific target folders and files to the current working directory
    for item in TARGETS_TO_MOVE:
        source_path = os.path.join(source_dir, item)
        dest_path = os.path.join(".", item)
        
        # Overwrite existing files/folders if the cell is run multiple times
        if os.path.exists(dest_path):
            if os.path.isdir(dest_path):
                shutil.rmtree(dest_path)
            else:
                os.remove(dest_path)
                
        shutil.move(source_path, dest_path)
        
    # Clean up the temporary ZIP file and extraction directory
    os.remove(TEMP_ZIP_PATH)
    shutil.rmtree(EXTRACT_DIR)
    print("\nSpatial data successfully loaded to Colab environment.")

def setup_environment():
    # Main workflow to execute the setup process
    download_repository()
    extract_and_organize_data()

setup_environment()

In [ ]:
# @title **Step 02. Interactive Point Adjustment with Multi-Base Maps, Roads**

import os
import io
import base64
import warnings
import geopandas as gpd
import osmnx as ox
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling, transform_bounds
import numpy as np
from shapely.geometry import Point, box
from PIL import Image
from ipyleaflet import Map, Marker, TileLayer, ImageOverlay, GeoData, DivIcon, LayersControl, FullScreenControl
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings('ignore')
ox.settings.use_cache = True

# Configuration
DISTRICT_NAME = "Chengalpattu"
DISTRICT_BOUNDARY_DIR = r"/content/District"
RICE_RASTER_PATH = r"/content/GT_Points/TN_RA_P3_2025.tif"
OUTPUT_DIR = r"/content/GT_Points"
SAVE_PATH = f"/content/{DISTRICT_NAME}_GT.gpkg"

OSM_FILTER = '["highway"~"primary|secondary"]'
# OSM_FILTER = '["highway"~"primary|secondary|tertiary"]'

GPKG_PATH = os.path.join(OUTPUT_DIR, f"{DISTRICT_NAME}_GT.gpkg")
BOUNDARY_PATH = os.path.join(DISTRICT_BOUNDARY_DIR, f"{DISTRICT_NAME}.gpkg")

TARGET_CRS = 32644
RASTER_OPACITY = 0.4
MASK_OPACITY = 1.00
ROAD_OPACITY = 0.5
INITIAL_POINT_SIZE = 12
DEFAULT_COLOR = '#3388ff'
ACTIVE_COLOR = '#ffbb00'

def generate_district_assets():
    if not os.path.exists(BOUNDARY_PATH):
        print(f"Error: Boundary file not found at {BOUNDARY_PATH}")
        return None, None, None, None

    district = gpd.read_file(BOUNDARY_PATH)
    district_utm = district.to_crs(epsg=TARGET_CRS)
    district_wgs84 = district.to_crs(epsg=4326)

    district_polygon = district_wgs84.geometry.union_all()

    # Generate inverted mask for exterior obscuration
    world_box = box(-180, -90, 180, 90)
    inverted_geom = world_box.difference(district_polygon)
    inverted_gdf = gpd.GeoDataFrame(geometry=[inverted_geom], crs=4326)

    print("Downloading OSM roads for visualization...")
    try:
        G_wgs84 = ox.graph_from_polygon(district_polygon, custom_filter=OSM_FILTER, simplify=True)
        _, edges_wgs84 = ox.graph_to_gdfs(G_wgs84)
    except Exception as e:
        print(f"Failed to fetch OSM roads: {e}")
        edges_wgs84 = gpd.GeoDataFrame(geometry=[], crs=4326)

    print("Generating raster overlay...")
    with rasterio.open(RICE_RASTER_PATH) as src:
        with WarpedVRT(src, crs=f"EPSG:{TARGET_CRS}") as vrt:
            out_image, out_transform = mask(vrt, district_utm.geometry, crop=True)

            height, width = out_image[0].shape
            left, bottom, right, top = rasterio.transform.array_bounds(height, width, out_transform)

            src_crs = f"EPSG:{TARGET_CRS}"
            dst_crs = "EPSG:4326"

            bounds_wgs84 = transform_bounds(src_crs, dst_crs, left, bottom, right, top)
            min_lon, min_lat, max_lon, max_lat = bounds_wgs84
            image_bounds = [[min_lat, min_lon], [max_lat, max_lon]]

            dst_transform, dst_width, dst_height = calculate_default_transform(
                src_crs, dst_crs, width, height, left, bottom, right, top
            )

            dst_array = np.zeros((1, dst_height, dst_width), dtype=np.uint8)
            reproject(
                source=out_image,
                destination=dst_array,
                src_transform=out_transform,
                src_crs=src_crs,
                dst_transform=dst_transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest
            )

            rgba = np.zeros((dst_height, dst_width, 4), dtype=np.uint8)
            rice_mask = dst_array[0] == 1
            rgba[..., 1] = np.where(rice_mask, 255, 0)
            rgba[..., 3] = np.where(rice_mask, 255, 0)

            img = Image.fromarray(rgba)
            img_byte_arr = io.BytesIO()
            img.save(img_byte_arr, format='PNG')
            encoded_img = base64.b64encode(img_byte_arr.getvalue()).decode('ascii')
            img_url = 'data:image/png;base64,' + encoded_img

            return img_url, image_bounds, inverted_gdf, edges_wgs84

def interactive_adjustment():
    load_path = SAVE_PATH if os.path.exists(SAVE_PATH) else GPKG_PATH
    
    if not os.path.exists(load_path):
        print(f"Error: Ground truth file not found at {load_path}.")
        return

    gt_gdf = gpd.read_file(load_path)
    gt_wgs84 = gt_gdf.to_crs(epsg=4326)

    img_url, image_bounds, mask_gdf, edges_wgs84 = generate_district_assets()
    if not img_url:
        return

    clear_output()

    initial_y = gt_wgs84.geometry.iloc[0].y if not gt_wgs84.empty else 0
    initial_x = gt_wgs84.geometry.iloc[0].x if not gt_wgs84.empty else 0

    # Configure base layers
    google_satellite = TileLayer(url='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}', name='Google Satellite', base=True)
    google_hybrid = TileLayer(url='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}', name='Google Hybrid', base=True)
    google_map = TileLayer(url='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}', name='Google Maps', base=True)

    # Configure analytical overlays
    rice_overlay = ImageOverlay(url=img_url, bounds=image_bounds, opacity=RASTER_OPACITY, name='Rice Extent')

    boundary_mask_layer = GeoData(
        geo_dataframe=mask_gdf,
        style={'color': 'black', 'weight': 3, 'fillColor': '#ffffff', 'fillOpacity': MASK_OPACITY},
        name='District Boundary Mask'
    )

    roads_layer = GeoData(
        geo_dataframe=edges_wgs84,
        style={'color': 'red', 'weight': 2, 'opacity': ROAD_OPACITY},
        name='OSM Roads'
    )

    m = Map(
        center=(initial_y, initial_x),
        zoom=12,
        layers=[google_hybrid, google_satellite, google_map, rice_overlay, boundary_mask_layer, roads_layer],
        scroll_wheel_zoom=True,
        layout=widgets.Layout(height='800px')
    )

    m.add_control(LayersControl())
    m.add_control(FullScreenControl())

    def create_custom_icon(size, color):
        html_str = f'<div style="background-color: {color}; border: 2px solid white; width: {size}px; height: {size}px; border-radius: 50%; box-shadow: 0 0 4px rgba(0,0,0,0.5);"></div>'
        return DivIcon(html=html_str, bg_pos=[0, 0], icon_size=[size, size], icon_anchor=[size//2, size//2])

    current_point_size = [INITIAL_POINT_SIZE]
    backgrounds_visible = [True]
    active_points = []
    current_index = [0]

    status_label = widgets.HTML(value="")
    output = widgets.Output()

    decrease_size_button = widgets.Button(description='Decrease Size', button_style='warning', icon='compress')
    increase_size_button = widgets.Button(description='Increase Size', button_style='warning', icon='expand')
    toggle_bg_button = widgets.Button(description='Toggle Overlays', button_style='primary', icon='eye-slash')
    add_point_toggle = widgets.ToggleButton(value=False, description='Add Point (Click Map)', button_style='primary', icon='plus')

    prev_button = widgets.Button(description='Previous Point', button_style='info', icon='arrow-left')
    next_button = widgets.Button(description='Next Point', button_style='info', icon='arrow-right')
    delete_button = widgets.Button(description='Delete Point', button_style='danger', icon='trash')
    save_button = widgets.Button(description='Save Adjusted Points', button_style='success', icon='check')

    def update_status(msg=""):
        if not active_points:
            status_label.value = f"<b>Status:</b> No points remaining. {msg}"
            return
        current_pt = active_points[current_index[0]]
        status_label.value = f"<b>Viewing Point:</b> {current_index[0] + 1} / {len(active_points)} &nbsp;&nbsp;|&nbsp;&nbsp; <b>FID:</b> {current_pt['fid']} &nbsp;&nbsp; {msg}"

    def auto_save():
        if not active_points:
            return
        geoms = [Point(pt['marker'].location[1], pt['marker'].location[0]) for pt in active_points]
        fids = [pt['fid'] for pt in active_points]
        adjusted_gdf_wgs84 = gpd.GeoDataFrame({'original_fid': fids, 'geometry': geoms}, crs=4326)
        final_gdf_utm = adjusted_gdf_wgs84.to_crs(epsg=TARGET_CRS)
        # Saves to the new /content path, leaving the raw file untouched
        final_gdf_utm.to_file(SAVE_PATH, driver="GPKG")
        update_status("<span style='color:green;'>(Saved to /content)</span>")

    def set_active_point(new_idx):
        if not active_points: return
        if current_index[0] < len(active_points):
            active_points[current_index[0]]['marker'].icon = create_custom_icon(current_point_size[0], DEFAULT_COLOR)
        current_index[0] = new_idx
        active_points[current_index[0]]['marker'].icon = create_custom_icon(current_point_size[0], ACTIVE_COLOR)
        m.center = active_points[current_index[0]]['marker'].location
        update_status()

    def update_all_icons():
        def_icon = create_custom_icon(current_point_size[0], DEFAULT_COLOR)
        act_icon = create_custom_icon(current_point_size[0], ACTIVE_COLOR)
        for i, pt in enumerate(active_points):
            if i == current_index[0]:
                pt['marker'].icon = act_icon
            else:
                pt['marker'].icon = def_icon

    def make_marker_click_handler(fid):
        def handler(**kwargs):
            for i, pt in enumerate(active_points):
                if pt['fid'] == fid:
                    set_active_point(i)
                    break
        return handler

    def on_location_change(change):
        # auto_save()
        pass

    for idx, row in gt_wgs84.iterrows():
        marker = Marker(
            location=(row.geometry.y, row.geometry.x),
            draggable=True,
            title=f"FID: {row['original_fid']}",
            icon=create_custom_icon(current_point_size[0], DEFAULT_COLOR)
        )
        marker.on_click(make_marker_click_handler(row['original_fid']))
        # marker.observe(on_location_change, names='location')
        m.add_layer(marker)
        active_points.append({'marker': marker, 'fid': row['original_fid']})

    if active_points:
        active_points[0]['marker'].icon = create_custom_icon(current_point_size[0], ACTIVE_COLOR)

    update_status()

    def handle_map_click(**kwargs):
        if kwargs.get('type') == 'click' and add_point_toggle.value:
            lat, lon = kwargs.get('coordinates')
            new_fid = max([pt['fid'] for pt in active_points] + [-1]) + 1

            new_marker = Marker(
                location=(lat, lon),
                draggable=True,
                title=f"FID: {new_fid}",
                icon=create_custom_icon(current_point_size[0], DEFAULT_COLOR)
            )
            new_marker.on_click(make_marker_click_handler(new_fid))
            # new_marker.observe(on_location_change, names='location')
            m.add_layer(new_marker)
            active_points.append({'marker': new_marker, 'fid': new_fid})

            add_point_toggle.value = False
            set_active_point(len(active_points) - 1)
            # auto_save()

    m.on_interaction(handle_map_click)

    def on_prev_button_clicked(b):
        if not active_points: return
        new_idx = (current_index[0] - 1) % len(active_points)
        set_active_point(new_idx)

    def on_next_button_clicked(b):
        if not active_points: return
        new_idx = (current_index[0] + 1) % len(active_points)
        set_active_point(new_idx)

    def on_delete_button_clicked(b):
        if not active_points: return
        idx = current_index[0]
        pt_to_remove = active_points.pop(idx)
        m.remove_layer(pt_to_remove['marker'])

        if not active_points:
            update_status()
            # auto_save()
            return

        if current_index[0] >= len(active_points):
            current_index[0] = 0

        active_points[current_index[0]]['marker'].icon = create_custom_icon(current_point_size[0], ACTIVE_COLOR)
        m.center = active_points[current_index[0]]['marker'].location
        # auto_save()

    def on_decrease_size_button_clicked(b):
        current_point_size[0] = max(4, current_point_size[0] - 2)
        update_all_icons()

    def on_increase_size_button_clicked(b):
        current_point_size[0] = min(40, current_point_size[0] + 2)
        update_all_icons()

    def on_toggle_bg_button_clicked(b):
        backgrounds_visible[0] = not backgrounds_visible[0]
        # Only toggle overlays so the selected base map remains visible
        rice_overlay.visible = backgrounds_visible[0]
        roads_layer.visible = backgrounds_visible[0]
        boundary_mask_layer.visible = backgrounds_visible[0]

        if backgrounds_visible[0]:
            toggle_bg_button.icon = 'eye-slash'
        else:
            toggle_bg_button.icon = 'eye'

    def on_save_button_clicked(b):
        auto_save()
        update_status("<span style='color:blue;'>(Manually Saved)</span>")

    prev_button.on_click(on_prev_button_clicked)
    next_button.on_click(on_next_button_clicked)
    delete_button.on_click(on_delete_button_clicked)
    save_button.on_click(on_save_button_clicked)
    decrease_size_button.on_click(on_decrease_size_button_clicked)
    increase_size_button.on_click(on_increase_size_button_clicked)
    toggle_bg_button.on_click(on_toggle_bg_button_clicked)

    view_controls = widgets.HBox([decrease_size_button, increase_size_button, toggle_bg_button, add_point_toggle])
    edit_controls = widgets.HBox([prev_button, next_button, delete_button, save_button])

    display(m, view_controls, status_label, edit_controls, output)

def execute_workflow():
    interactive_adjustment()

execute_workflow()